# Time Shifting, Differencing, and Returns

Three fundamental operations transform time series data in pandas: **`shift`**,
**`diff`**, and **`pct_change`**. Together they are the building blocks for:

- **Feature engineering** — creating lag features as inputs for autoregressive models
- **Stationarity transforms** — removing trend so that statistical models can work properly
- **Return calculations** — measuring period-over-period growth rates

This notebook walks through each operation, shows how they relate to one another
mathematically, and demonstrates practical use cases with real data.

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path("../data")

print(f"pandas      {pd.__version__}")
print(f"numpy       {np.__version__}")

In [ ]:
# Airline passengers — classic monthly time series with trend + seasonality
df_air = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    parse_dates=True,
    index_col="Month",
)
df_air.index.freq = pd.infer_freq(df_air.index)

print(f"Shape: {df_air.shape}")
print(f"Range: {df_air.index.min()} → {df_air.index.max()}")
print(f"Freq:  {df_air.index.freq}")
df_air.head()

In [ ]:
# Alcohol sales — monthly series with strong seasonality
df_alc = pd.read_csv(
    DATA_DIR / "Alcohol_Sales.csv",
    parse_dates=True,
    index_col="DATE",
)
df_alc.index.freq = pd.infer_freq(df_alc.index)

print(f"Shape: {df_alc.shape}")
print(f"Range: {df_alc.index.min()} → {df_alc.index.max()}")
print(f"Freq:  {df_alc.index.freq}")
df_alc.head()

In [ ]:
# Convenience: pull out the main column as a Series for each dataset
passengers = df_air['Thousands of Passengers']
alcohol = df_alc['S4248SM144NCEN']

---
## 2. `shift()` — Creating Lag and Lead Features

The `shift()` method moves data **up or down** along the index without changing
the index itself. This is the fundamental operation for creating **lag features**.

- `shift(1)` — shifts data **down** by 1 period (lag-1: previous observation)
- `shift(-1)` — shifts data **up** by 1 period (lead-1: next observation)
- `shift(12)` — lag-12 for monthly data (same month last year)

Mathematically, if the original series is $y_t$, then `shift(k)` gives $y_{t-k}$.

In [ ]:
# shift(1): lag-1 — each value becomes the PREVIOUS month's value
lag1 = passengers.shift(1)

pd.DataFrame({
    'original': passengers,
    'shift(1) [lag-1]': lag1,
}).head(8)

In [ ]:
# shift(-1): lead-1 — each value becomes the NEXT month's value
lead1 = passengers.shift(-1)

pd.DataFrame({
    'original': passengers,
    'shift(-1) [lead-1]': lead1,
}).tail(8)

In [ ]:
# shift(12): lag-12 — same month from the previous year
lag12 = passengers.shift(12)

pd.DataFrame({
    'original': passengers,
    'shift(12) [lag-12]': lag12,
}).iloc[10:22]  # show where the first non-NaN lag-12 appears

In [ ]:
# Visualize: original vs lag-1 and lag-12
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Lag-1
axes[0].plot(passengers, label='Original', linewidth=1.5)
axes[0].plot(passengers.shift(1), label='shift(1) — lag-1',
             linewidth=1.5, linestyle='--', alpha=0.7)
axes[0].set_title('Original vs Lag-1')
axes[0].set_ylabel('Thousands of Passengers')
axes[0].legend()

# Lag-12
axes[1].plot(passengers, label='Original', linewidth=1.5)
axes[1].plot(passengers.shift(12), label='shift(12) — lag-12',
             linewidth=1.5, linestyle='--', alpha=0.7)
axes[1].set_title('Original vs Lag-12 (same month last year)')
axes[1].legend()

plt.tight_layout()
plt.show()

**Why this matters:** Lag features are the core inputs for **autoregressive (AR)
models**. An AR(1) model predicts $y_t$ from $y_{t-1}$; an AR(p) model uses
$y_{t-1}, y_{t-2}, \dots, y_{t-p}$. The `shift()` method is how you build
those lagged columns in practice.

---
## 3. `diff()` — Differencing

Differencing computes the change between consecutive observations. It is the
primary tool for **removing trend** from a time series.

- **First difference:** $\Delta y_t = y_t - y_{t-1}$ &nbsp; → &nbsp; `diff(1)`
- **Seasonal difference:** $y_t - y_{t-12}$ &nbsp; → &nbsp; `diff(12)` (for monthly data)
- **Second-order difference:** $\Delta^2 y_t = \Delta(\Delta y_t)$ &nbsp; → &nbsp; `diff().diff()`

`diff(k)` is mathematically equivalent to `series - series.shift(k)`.

In [ ]:
# First difference: y_t - y_{t-1}
diff1 = passengers.diff(1)

# Verify it equals series - shift(1)
manual_diff = passengers - passengers.shift(1)
print("diff(1) equals (series - shift(1)):", diff1.equals(manual_diff))

pd.DataFrame({
    'original': passengers,
    'diff(1)': diff1,
    'manual: y - shift(1)': manual_diff,
}).head(8)

In [ ]:
# Seasonal difference: y_t - y_{t-12}
diff12 = passengers.diff(12)

pd.DataFrame({
    'original': passengers,
    'diff(12) [seasonal]': diff12,
}).iloc[10:22]

In [ ]:
# Second-order differencing: diff of the diff
diff2 = passengers.diff().diff()

pd.DataFrame({
    'original': passengers,
    'diff(1)': diff1,
    'diff().diff() [2nd order]': diff2,
}).head(8)

In [ ]:
# Visualize: original vs differenced series
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(passengers, linewidth=1.5, color='steelblue')
axes[0].set_title('Original — Airline Passengers')
axes[0].set_ylabel('Passengers (thousands)')

axes[1].plot(diff1, linewidth=1.2, color='darkorange')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('First Difference — $y_t - y_{t-1}$')
axes[1].set_ylabel('Change')

axes[2].plot(diff12, linewidth=1.2, color='firebrick')
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_title('Seasonal Difference — $y_t - y_{t-12}$')
axes[2].set_ylabel('Change vs same month last year')

plt.tight_layout()
plt.show()

Notice how the **first difference** removes the upward trend — the differenced
series fluctuates around zero. The **seasonal difference** removes the
repeating annual pattern. This is exactly what we need before fitting
ARIMA-family models.

**Why this matters:** The "I" in **ARIMA** stands for *Integrated*, meaning
the series has been differenced $d$ times to achieve stationarity. The
parameter $d$ in ARIMA$(p, d, q)$ is the order of differencing.

---
## 4. `pct_change()` — Percentage Returns

The `pct_change()` method computes the **fractional change** between periods:

$$\text{pct\_change}(k) = \frac{y_t - y_{t-k}}{y_{t-k}}$$

This is equivalent to `diff(k) / shift(k)`. Multiply by 100 to get a
percentage.

In [ ]:
# Month-over-month percentage change
pct1 = passengers.pct_change(1)

# Verify equivalence
manual_pct = passengers.diff(1) / passengers.shift(1)
print("pct_change(1) equals diff(1)/shift(1):",
      np.allclose(pct1.dropna(), manual_pct.dropna()))

pd.DataFrame({
    'original': passengers,
    'pct_change(1)': pct1,
    'as percentage': pct1 * 100,
}).head(8)

In [ ]:
# Year-over-year percentage change (period=12 for monthly data)
pct12 = passengers.pct_change(12)

pd.DataFrame({
    'original': passengers,
    'YoY pct_change': pct12,
    'YoY %': (pct12 * 100).round(2),
}).iloc[10:22]

In [ ]:
# Visualize pct_change as a bar chart
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Month-over-month
colors_mom = ['firebrick' if v < 0 else 'seagreen' for v in pct1.dropna()]
axes[0].bar(pct1.dropna().index, pct1.dropna() * 100, color=colors_mom, width=25)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Month-over-Month % Change — Airline Passengers')
axes[0].set_ylabel('% Change')

# Year-over-year
colors_yoy = ['firebrick' if v < 0 else 'steelblue' for v in pct12.dropna()]
axes[1].bar(pct12.dropna().index, pct12.dropna() * 100, color=colors_yoy, width=25)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Year-over-Year % Change — Airline Passengers')
axes[1].set_ylabel('% Change')

plt.tight_layout()
plt.show()

---
## 5. Combining Operations

These three methods combine naturally for common time series tasks.

### 5a. Year-over-Year Growth Rate

In [ ]:
# Year-over-year growth as a percentage
yoy_growth = passengers.pct_change(12) * 100

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(yoy_growth, linewidth=1.5, color='darkorange')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.fill_between(yoy_growth.index, yoy_growth, 0,
                where=yoy_growth >= 0, alpha=0.3, color='seagreen')
ax.fill_between(yoy_growth.index, yoy_growth, 0,
                where=yoy_growth < 0, alpha=0.3, color='firebrick')
ax.set_title('Year-over-Year Growth Rate (%) — Airline Passengers')
ax.set_ylabel('Growth %')
plt.tight_layout()
plt.show()

### 5b. Log Returns

Log returns are widely used in finance because they are **additive** across
time periods:

$$r_t = \ln\left(\frac{y_t}{y_{t-1}}\right) = \ln(y_t) - \ln(y_{t-1})$$

In [ ]:
# Log returns
log_returns = np.log(passengers / passengers.shift(1))

# Alternative: np.log(series).diff()
log_returns_alt = np.log(passengers).diff()
print("Both approaches equal:",
      np.allclose(log_returns.dropna(), log_returns_alt.dropna()))

pd.DataFrame({
    'original': passengers,
    'log_return': log_returns,
    'pct_change': passengers.pct_change(1),
}).head(8)

### 5c. Recovering the Original via Cumulative Sum

Since differencing removes information (the starting value), we can recover
the original series from its differences using `cumsum()` plus the initial value:

$$y_t = y_0 + \sum_{i=1}^{t} \Delta y_i$$

In [ ]:
# Recover original from first differences + initial value
initial_value = passengers.iloc[0]
recovered = passengers.diff().cumsum() + initial_value

# Verify recovery (ignoring the first NaN)
print("Recovered matches original:",
      np.allclose(passengers.iloc[1:], recovered.iloc[1:]))

pd.DataFrame({
    'original': passengers,
    'recovered': recovered,
}).head(8)

---
## 6. Building a Feature Matrix

In practice, forecasting models need a **feature matrix** where each row
corresponds to a time step and each column is a feature. Lag features,
differences, and percentage changes are the most common engineered features
for time series.

In [ ]:
# Build a feature matrix for the airline passengers dataset
features = pd.DataFrame({
    'y': passengers,
    'lag_1': passengers.shift(1),
    'lag_2': passengers.shift(2),
    'lag_12': passengers.shift(12),
    'diff_1': passengers.diff(1),
    'diff_12': passengers.diff(12),
    'pct_change_1': passengers.pct_change(1),
    'pct_change_12': passengers.pct_change(12),
})

print(f"Shape: {features.shape}")
print(f"NaN per column:\n{features.isna().sum()}")
print()
features.head(15)

In [ ]:
# Drop rows with NaN (common before training a model)
features_clean = features.dropna()
print(f"Rows before dropna: {len(features)}")
print(f"Rows after  dropna: {len(features_clean)}")
print(f"Rows lost:          {len(features) - len(features_clean)}")
print()
features_clean.head(10)

In [ ]:
# Same idea with the Alcohol Sales dataset — more lags via a loop
feat_alc = pd.DataFrame({'y': alcohol})
for lag in [1, 2, 3, 6, 12]:
    feat_alc[f'lag_{lag}'] = alcohol.shift(lag)
    feat_alc[f'diff_{lag}'] = alcohol.diff(lag)

feat_alc.dropna().head(10)

---
## 7. Practical Example — Alcohol Sales

Let's apply all three operations to the Alcohol Sales dataset and visualize
the results side by side.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

# Original
axes[0].plot(alcohol, linewidth=1.2, color='steelblue')
axes[0].set_title('Alcohol Sales — Original')
axes[0].set_ylabel('Sales ($M)')

# First difference
axes[1].plot(alcohol.diff(1), linewidth=1, color='darkorange')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('First Difference — $y_t - y_{t-1}$')
axes[1].set_ylabel('Change')

# Seasonal difference
axes[2].plot(alcohol.diff(12), linewidth=1, color='firebrick')
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_title('Seasonal Difference — $y_t - y_{t-12}$')
axes[2].set_ylabel('Change vs last year')

# Year-over-year % change
yoy_alc = alcohol.pct_change(12) * 100
axes[3].plot(yoy_alc, linewidth=1, color='seagreen')
axes[3].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[3].set_title('Year-over-Year % Change')
axes[3].set_ylabel('% Change')

plt.tight_layout()
plt.show()

---
## 8. Summary

| Method | Formula | What It Does | Key Use Case |
|---|---|---|---|
| `shift(k)` | $y_{t-k}$ | Moves values up/down without changing the index | Creating **lag features** for AR models |
| `shift(-k)` | $y_{t+k}$ | Lead — future values | Lookahead targets (use carefully to avoid data leakage) |
| `diff(k)` | $y_t - y_{t-k}$ | Difference between current and $k$-lagged value | **Removing trend** — the "I" in ARIMA |
| `diff().diff()` | $\Delta^2 y_t$ | Second-order differencing | Removing quadratic trend |
| `pct_change(k)` | $(y_t - y_{t-k}) / y_{t-k}$ | Fractional change over $k$ periods | **Growth rates**, returns |
| `np.log(y/y.shift(1))` | $\ln(y_t / y_{t-1})$ | Log return (additive over time) | **Financial returns** |
| `diff().cumsum()` | $y_0 + \sum \Delta y_i$ | Cumulative sum of differences | **Recovering** the original series |

### Key Relationships

```
diff(k)       ==  series - shift(k)
pct_change(k) ==  diff(k) / shift(k)
cumsum(diff()) + y_0  ==  original series
```

### When to Use What

- **`shift()`** when you need past (or future) values as separate columns
  (feature engineering)
- **`diff()`** when you need to make a series **stationary** by removing trend
  or seasonality
- **`pct_change()`** when you care about **relative** change rather than
  absolute change (e.g., growth rates, investment returns)